<a href="https://colab.research.google.com/github/Mantulisme/SourceCode_ABSA/blob/main/steam_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import time
import os
import json
from datetime import datetime

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Scraping the data

In [ ]:
def get_game_name(appid):
    """Fetch the game name from Steam API"""
    try:
        url = f"https://store.steampowered.com/api/appdetails"
        params = {'appids': appid}
        response = requests.get(url, params=params, timeout=10)
        data = response.json()

        if str(appid) in data and data[str(appid)]['success']:
            game_name = data[str(appid)]['data']['name']
            game_name = game_name.replace(' ', '_').replace(':', '').replace('/', '_')
            game_name = ''.join(c for c in game_name if c.isalnum() or c == '_')
            return game_name
        else:
            return f"appid_{appid}"
    except Exception as e:
        print(f"Could not fetch game name: {e}")
        return f"appid_{appid}"

In [ ]:
def scrape_steam_reviews(appid, filter_type='all', language='english', max_reviews=None):
    """
    Scrape Steam reviews for a given app ID

    Args:
        appid: Steam app ID
        filter_type: 'all', 'recent', 'updated'
        language: language code (e.g., 'english', 'spanish')
        max_reviews: maximum number of reviews to fetch (None = all available)

    Returns:
        List of review dictionaries
    """
    reviews = []
    cursor = "*"
    request_count = 0

    print(f"Starting to scrape reviews for App ID: {appid}")
    print(f"Filter: {filter_type}, Language: {language}\n")

    while True:
        url = f"https://store.steampowered.com/appreviews/{appid}"
        params = {
            'json': 1,
            'filter': filter_type,
            'language': language,
            'cursor': cursor,
            'num_per_page': 100
        }

        try:
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            request_count += 1

            if not data.get('success', 0):
                print(f"API returned success=False. Stopping.")
                break

            batch_reviews = data.get('reviews', [])

            if not batch_reviews:
                print("No more reviews to fetch.")
                break

            reviews.extend(batch_reviews)

            cursor = data.get('cursor')

            print(f"Request {request_count}: Fetched {len(batch_reviews)} reviews (Total: {len(reviews)})")

            if max_reviews and len(reviews) >= max_reviews:
                reviews = reviews[:max_reviews]
                print(f"\nReached maximum of {max_reviews} reviews.")
                break

            time.sleep(1)

        except requests.exceptions.RequestException as e:
            print(f"Error fetching reviews: {e}")
            break
        except json.JSONDecodeError as e:
            print(f"Error parsing JSON: {e}")
            break

    return reviews

In [ ]:
def save_reviews_to_json(reviews, filename, folder='steam_reviews'):
    """Save reviews to a JSON file in a specified folder"""
    os.makedirs(folder, exist_ok=True)

    filepath = os.path.join(folder, filename)

    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(reviews, f, indent=2, ensure_ascii=False)
    print(f"\nSaved {len(reviews)} reviews to {filepath}")

In [ ]:
def extract_key_fields(reviews):
    """Extract key fields from reviews for easier analysis"""
    simplified = []
    for review in reviews:
        simplified.append({
            'recommendationid': review.get('recommendationid'),
            'author_steamid': review.get('author', {}).get('steamid'),
            'language': review.get('language'),
            'review_text': review.get('review'),
            'timestamp_created': review.get('timestamp_created'),
            'timestamp_updated': review.get('timestamp_updated'),
            'voted_up': review.get('voted_up'),
        })
    return simplified

In [ ]:
if __name__ == "__main__":
    APPID = input("Game App ID: ")

    reviews = scrape_steam_reviews(
        appid=APPID,
        filter_type='all',
        language='english',
        max_reviews=100
    )

    print(f"\n{'='*60}")
    print(f"Total reviews scraped: {len(reviews)}")
    print(f"{'='*60}")

    game_name = get_game_name(APPID)
    print(f"Game name: {game_name}")

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_reviews_to_json(reviews, f'{game_name}_reviews_full_{timestamp}.json')

    simplified_reviews = extract_key_fields(reviews)
    save_reviews_to_json(simplified_reviews, f'{game_name}_reviews_simplified_{timestamp}.json')

    if reviews:
        positive = sum(1 for r in reviews if r.get('voted_up'))
        negative = len(reviews) - positive
        print(f"\nReview Statistics:")
        print(f"Positive reviews: {positive} ({positive/len(reviews)*100:.1f}%)")
        print(f"Negative reviews: {negative} ({negative/len(reviews)*100:.1f}%)")
        print(f"\nSample review text:")
        print(f"{reviews[0].get('review', '')[:200]}...")

Game App ID: 3450310
Starting to scrape reviews for App ID: 3450310
Filter: all, Language: english

Request 1: Fetched 100 reviews (Total: 100)

Reached maximum of 100 reviews.

Total reviews scraped: 100
Game name: Europa_Universalis_V

Saved 100 reviews to steam_reviews/Europa_Universalis_V_reviews_full_20260326_032034.json

Saved 100 reviews to steam_reviews/Europa_Universalis_V_reviews_simplified_20260326_032034.json

Review Statistics:
Positive reviews: 16 (16.0%)
Negative reviews: 84 (84.0%)

Sample review text:
Veteran Europa Universalis player here (5000+ hrs) : I have never left a negative review for a game in this series before- but after the most recent update to EUV I feel I have to speak up. 

I really...


# Splitting the data

In [ ]:
import glob
import pandas as pd

In [ ]:
csv_folder = "/content/drive/MyDrive/Dataset steam/New_2025/csv"
n_per_game = 5

all_samples = []

for csv_path in glob.glob(os.path.join(csv_folder, "*.csv")):
    df = pd.read_csv(csv_path)
    sample = df.sample(n=n_per_game, random_state=42)
    sample["game"] = os.path.splitext(os.path.basename(csv_path))[0]
    all_samples.append(sample)

reference_df = pd.concat(all_samples, ignore_index=True)
reference_df.to_csv("reference_120.csv", index=False)
print(f"Total samples: {len(reference_df)}")

Total samples: 120


In [ ]:
dropped_columns = ['author_steamid',
                   'author_steamid',
                   'author_playtime_forever',
                   'author_playtime_at_review',
                   'language', 'timestamp_created',
                   'timestamp_updated', 'voted_up',
                   'votes_up', 'votes_funny',
                   'weighted_vote_score',
                   'comment_count',
                   'steam_purchase',
                   'received_for_free',
                   'written_during_early_access']

In [ ]:
df_ref = pd.read_csv("reference_120.csv")
df_ref = df_ref.drop(columns=dropped_columns)
df_ref.to_csv("cleaned120_ref.csv", index=False)